## Example 1: Run model-written Python in Monty's sandbox

In [ ]:
import importlib.metadata

from agent_framework_monty import MontyExecuteCodeTool

# Example 1: the execute_code tool runs code in pydantic-monty (a Rust Python interpreter), not in this process
assert importlib.metadata.version("agent-framework-monty") == "1.0.0b261106"
assert importlib.metadata.version("pydantic-monty") == "1.0.0"
tool = MontyExecuteCodeTool()
contents = await tool.invoke(arguments={"code": "print(sum(i * i for i in range(10)))"})
assert [(c.type, c.text) for c in contents] == [("text", "285\n")], [(c.type, c.text) for c in contents]
print("Example 1 OK:", tool.name)

## Example 2: Code calls the agent's tools

In [ ]:
from agent_framework_monty import MontyExecuteCodeTool

def get_price(item: str) -> float:
    """Price of one item, in euros."""
    return {"apple": 0.5, "pear": 0.75}[item]

# Example 2: CodeAct lets one model turn call several tools and combine the results in code; inside the sandbox a tool
# is an async function called with keyword arguments, and the code is type-checked against its signature first
tool = MontyExecuteCodeTool(tools=[get_price])
code = "apple = await get_price(item='apple')\npear = await get_price(item='pear')\nprint(apple * 4 + pear * 2)"
contents = await tool.invoke(arguments={"code": code})
assert [(c.type, c.text) for c in contents] == [("text", "3.5\n")], [(c.type, c.text) for c in contents]
print("Example 2 OK:", contents[0].text.strip())

## Example 3: Resource limits: the 1.x names and the legacy one

In [ ]:
from agent_framework_monty import MontyExecuteCodeTool

# Example 3: max_duration_secs (pydantic-monty 0.x) is mapped onto max_feed_duration_secs (1.x) by this release
for limits in ({"max_duration_secs": 5.0}, {"max_feed_duration_secs": 5.0}, {"max_turn_duration_secs": 5.0}):
    tool = MontyExecuteCodeTool(resource_limits=dict(limits))
    contents = await tool.invoke(arguments={"code": "print('done')"})
    assert contents[0].text == "done\n" and tool.resource_limits == limits, (limits, contents[0].text)
both = MontyExecuteCodeTool(resource_limits={"max_duration_secs": 5.0, "max_feed_duration_secs": 1.0}, include_detailed_errors=True)
contents = await both.invoke(arguments={"code": "print('must not run')"})
assert contents[0].type == "error" and "Specify only one of" in contents[0].error_details, contents[0].error_details
print("Example 3 OK")

## Example 4: A runaway loop is stopped by the time limit

In [ ]:
from agent_framework_monty import MontyExecuteCodeTool

# Example 4: the sandbox enforces the limit; the error comes back to the model as content, not as a crash
tool = MontyExecuteCodeTool(resource_limits={"max_feed_duration_secs": 0.05}, include_detailed_errors=True)
contents = await tool.invoke(arguments={"code": "while True:\n    pass"})
assert [c.type for c in contents] == ["error"], [c.type for c in contents]
assert "feed time limit exceeded" in contents[0].error_details, contents[0].error_details
print("Example 4 OK:", contents[0].error_details)

## Example 5: The context provider and agent_framework.monty

In [ ]:
import agent_framework.monty as afm

from agent_framework_monty import MontyCodeActProvider, MontyExecuteCodeTool

# Example 5: agent_framework re-exports this package; the provider adds the tool and its instructions to an agent
assert afm.MontyCodeActProvider is MontyCodeActProvider
provider = MontyCodeActProvider(tools=[len])
assert provider.source_id == "monty_codeact" and [t.name for t in provider.get_tools()] == ["len"], provider.get_tools()
instructions = MontyExecuteCodeTool(tools=[len]).build_instructions(tools_visible_to_model=False)
assert "execute_code" in instructions and "len" in instructions, instructions[:200]
print("Example 5 OK")